Leitura das tabelas da camada Silver

In [0]:
df_passagens = spark.table(
    "viagens_teste.silver.passagens"
)

df_hospedagens = spark.table(
    "viagens_teste.silver.hospedagens"
)

df_uber = spark.table(
    "viagens_teste.silver.uber"
)

df_diarias = spark.table(
    "viagens_teste.silver.diarias"
)

Validação das fontes da camada Silver

In [0]:
print("Passagens:", df_passagens.count())
print("Hospedagens:", df_hospedagens.count())
print("Uber:", df_uber.count())
print("Diárias:", df_diarias.count())

Padronização de Passagens para a camada Gold

In [0]:
from pyspark.sql.functions import col

df_gold_passagens = df_passagens.select(
    col("id_solicitacao").alias("id_registro"),
    col("viajante"),
    col("data_viagem").alias("data_evento"),
    col("centro_custo"),
    col("diretoria"),
    col("valor_numerico").alias("valor"),
    col("tipo_despesa"),
    col("trimestre"),
    col("agencia"),
    col("situacao")
)

Validação da estrutura Gold de Passagens

In [0]:
df_gold_passagens.printSchema()

display(df_gold_passagens)

In [0]:
df_gold_passagens.count()

Padronização de Hospedagens para a camada Gold

In [0]:
df_gold_hospedagens = df_hospedagens.select(
    col("id_hospedagem").alias("id_registro"),
    col("viajante"),
    col("data_checkin").alias("data_evento"),
    col("centro_custo"),
    col("diretoria"),
    col("valor_numerico").alias("valor"),
    col("tipo_despesa"),
    col("trimestre"),
    col("agencia"),
    col("situacao")
)

Validação da estrutura Gold de Hospedagens

In [0]:
df_gold_hospedagens.printSchema()

print(
    df_gold_passagens.columns ==
    df_gold_hospedagens.columns
)

In [0]:
print(
    "Hospedagens:",
    df_gold_hospedagens.count()
)

Padronização de Uber para a camada Gold

In [0]:
from pyspark.sql.functions import col, lit

df_gold_uber = df_uber.select(
    col("id_corrida").alias("id_registro"),
    col("viajante"),
    col("data_corrida").alias("data_evento"),
    col("centro_custo"),
    col("diretoria"),
    col("valor_numerico").alias("valor"),
    col("tipo_despesa"),
    col("trimestre"),
    col("agencia"),
    lit(None).cast("string").alias("situacao")
)

Validação da estrutura Gold de Uber

In [0]:
df_gold_uber.printSchema()

print(
    df_gold_passagens.columns ==
    df_gold_uber.columns
)

In [0]:
print(
    "Uber:",
    df_gold_uber.count()
)

Padronização de Diárias para a camada Gold

In [0]:
df_gold_diarias = df_diarias.select(
    col("id_diaria").alias("id_registro"),
    col("viajante"),
    col("data_inicio").alias("data_evento"),
    col("centro_custo"),
    col("diretoria"),
    col("valor_numerico").alias("valor"),
    col("tipo_despesa"),
    col("trimestre"),
    col("agencia"),
    lit(None).cast("string").alias("situacao")
)

Validação da estrutura Gold de Diárias

In [0]:
df_gold_diarias.printSchema()

print(
    df_gold_passagens.columns ==
    df_gold_diarias.columns
)

In [0]:
print(
    "Diárias:",
    df_gold_diarias.count()
)

Consolidação da tabela fato de viagens

In [0]:
df_fato_viagens = (
    df_gold_passagens
    .unionByName(df_gold_hospedagens)
    .unionByName(df_gold_uber)
    .unionByName(df_gold_diarias)
)

In [0]:
print("Total da fato_viagens:", df_fato_viagens.count())

Validação da consolidação

In [0]:
df_fato_viagens.groupBy(
    "tipo_despesa"
).count().show()

In [0]:
from pyspark.sql.functions import sum

df_fato_viagens.groupBy(
    "tipo_despesa"
).agg(
    sum("valor").alias("valor_total")
).show()

Persistência da tabela fato na camada Gold

In [0]:
df_fato_viagens.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("viagens_teste.gold.fato_viagens")

In [0]:
df_fato_teste = spark.table(
    "viagens_teste.gold.fato_viagens"
)

display(df_fato_teste)

In [0]:
print("Registros salvos:", df_fato_teste.count())

Criação da dimensão de centro de custo

In [0]:
df_dim_centro_custo = (
    df_fato_viagens
    .select(
        "centro_custo",
        "diretoria"
    )
    .distinct()
)

In [0]:
display(df_dim_centro_custo)

print(
    "Centros de custo:",
    df_dim_centro_custo.count()
)

Persistência da dimensão de centro de custo

In [0]:
df_dim_centro_custo.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "viagens_teste.gold.dim_centro_custo"
    )

In [0]:
display(
    spark.table(
        "viagens_teste.gold.dim_centro_custo"
    )
)

Criação da dimensão de agência

In [0]:
df_dim_agencia = (
    df_fato_viagens
    .select("agencia")
    .distinct()
)

In [0]:
display(df_dim_agencia)

In [0]:
df_dim_agencia.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("viagens_teste.gold.dim_agencia")

In [0]:
display(
    spark.table(
        "viagens_teste.gold.dim_agencia"
    )
)

Criação da dimensão de trimestre

In [0]:
df_dim_trimestre = (
    df_fato_viagens
    .select("trimestre")
    .distinct()
)

In [0]:
display(df_dim_trimestre)

Persistência da dimensão de trimestre

In [0]:
df_dim_trimestre.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("viagens_teste.gold.dim_trimestre")

In [0]:
display(
    spark.table(
        "viagens_teste.gold.dim_trimestre"
    )
)

In [0]:
dados_trimestre = [
    ("Primeiro Trimestre", 1),
    ("Segundo Trimestre", 2),
    ("Terceiro Trimestre", 3),
    ("Quarto Trimestre", 4)
]

df_dim_trimestre = spark.createDataFrame(
    dados_trimestre,
    ["trimestre", "ordem_trimestre"]
)

In [0]:
df_dim_trimestre.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("viagens_teste.gold.dim_trimestre")

Atualização da estrutura da dimensão de trimestre

In [0]:
spark.sql("""
DROP TABLE IF EXISTS viagens_teste.gold.dim_trimestre
""")

In [0]:
dados_trimestre = [
    ("Primeiro Trimestre", 1),
    ("Segundo Trimestre", 2),
    ("Terceiro Trimestre", 3),
    ("Quarto Trimestre", 4)
]

df_dim_trimestre = spark.createDataFrame(
    dados_trimestre,
    ["trimestre", "ordem_trimestre"]
)

In [0]:
df_dim_trimestre.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("viagens_teste.gold.dim_trimestre")

In [0]:
display(
    spark.table(
        "viagens_teste.gold.dim_trimestre"
    )
)

Criação da dimensão de tipo de despesa

In [0]:
df_dim_tipo_despesa = (
    df_fato_viagens
    .select("tipo_despesa")
    .distinct()
)

In [0]:
display(df_dim_tipo_despesa)

Persistência da dimensão de tipo de despesa

In [0]:
df_dim_tipo_despesa.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "viagens_teste.gold.dim_tipo_despesa"
    )

In [0]:
display(
    spark.table(
        "viagens_teste.gold.dim_tipo_despesa"
    )
)

Validação geral da camada Gold

In [0]:
print(
    "Fato viagens:",
    spark.table("viagens_teste.gold.fato_viagens").count()
)

print(
    "Centros de custo:",
    spark.table("viagens_teste.gold.dim_centro_custo").count()
)

print(
    "Agências:",
    spark.table("viagens_teste.gold.dim_agencia").count()
)

print(
    "Trimestres:",
    spark.table("viagens_teste.gold.dim_trimestre").count()
)

print(
    "Tipos de despesa:",
    spark.table("viagens_teste.gold.dim_tipo_despesa").count()
)

Validação financeira da tabela fato

In [0]:
from pyspark.sql.functions import sum

df_fato_viagens.groupBy(
    "tipo_despesa"
).agg(
    sum("valor").alias("valor_total")
).show()

Verificação de campos essenciais

In [0]:
df_fato_viagens.filter(
    col("centro_custo").isNull() |
    col("diretoria").isNull() |
    col("valor").isNull() |
    col("data_evento").isNull()
).show(truncate=False)

Criação da dimensão de datas

In [0]:
df_dim_data = spark.sql("""
SELECT
    data AS data,
    year(data) AS ano,
    month(data) AS numero_mes,

    CASE month(data)
        WHEN 1 THEN 'Janeiro'
        WHEN 2 THEN 'Fevereiro'
        WHEN 3 THEN 'Março'
        WHEN 4 THEN 'Abril'
        WHEN 5 THEN 'Maio'
        WHEN 6 THEN 'Junho'
        WHEN 7 THEN 'Julho'
        WHEN 8 THEN 'Agosto'
        WHEN 9 THEN 'Setembro'
        WHEN 10 THEN 'Outubro'
        WHEN 11 THEN 'Novembro'
        WHEN 12 THEN 'Dezembro'
    END AS mes,

    date_format(data, 'yyyy-MM') AS ano_mes,

    quarter(data) AS numero_trimestre,

    CASE quarter(data)
        WHEN 1 THEN 'Primeiro Trimestre'
        WHEN 2 THEN 'Segundo Trimestre'
        WHEN 3 THEN 'Terceiro Trimestre'
        WHEN 4 THEN 'Quarto Trimestre'
    END AS trimestre

FROM (
    SELECT explode(
        sequence(
            (SELECT min(data_evento) FROM viagens_teste.gold.fato_viagens),
            (SELECT max(data_evento) FROM viagens_teste.gold.fato_viagens),
            interval 1 day
        )
    ) AS data
)
""")

Validação da dimensão de datas

In [0]:
display(df_dim_data)

Persistência da dimensão de datas

In [0]:
df_dim_data.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("viagens_teste.gold.dim_data")

In [0]:
display(
    spark.table(
        "viagens_teste.gold.dim_data"
    )
)

In [0]:
%sql

SHOW TABLES IN viagens_teste.gold;

In [0]:
spark.catalog.tableExists(
    "viagens_teste.gold.dim_data"
)

In [0]:
%sql
SHOW TABLES IN viagens_teste.gold

In [0]:
%sql
SELECT *
FROM viagens_teste.gold.dim_data
LIMIT 10;

In [0]:
%sql
REFRESH TABLE viagens_teste.gold.dim_data;

In [0]:
%sql
SHOW TABLES IN viagens_teste.gold;

In [0]:
%sql
SELECT COUNT(*)
FROM viagens_teste.gold.dim_data;